# Практическая работа №3

## Сборка пайплайна предобработки

Датасет: Titanic

Цель: обработать пропущенные значения,
закодировать категориальные признаки,
масштабировать числовые данные
и предотвратить утечку информации.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

In [2]:
df = pd.read_csv("../data/titanic.csv")

display(df.head())

print("Размер таблицы:", df.shape)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


Размер таблицы: (891, 12)


In [3]:
numeric_features = ["Age", "Fare", "SibSp", "Parch"]
categorical_features = ["Pclass", "Sex", "Embarked"]

X = df[numeric_features + categorical_features].copy()
y = df["Survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (712, 7)
Test: (179, 7)


### Разделение данных

Датасет разделён на обучающую (80%) и тестовую (20%) выборки. Разделение выполняется до расчёта статистик, чтобы избежать утечки данных. Также используется стратификация для сохранения соотношения выживших и погибших.

In [4]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

### Обработка числовых признаков

Для Age, Fare, SibSp и Parch используется заполнение пропусков медианой. Медиана выбрана потому, что она менее чувствительна к выбросам, чем среднее. После заполнения применяется StandardScaler для масштабирования значений.

In [5]:
categorical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="constant",
            fill_value="Unknown"
        )
    ),
    (
        "encoder",
        OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        )
    )
])

### Обработка категориальных признаков

Для Sex, Pclass и Embarked применяется OneHotEncoder. Пропущенные категории заменяются на Unknown. Параметр handle_unknown="ignore" позволяет обрабатывать категории, которые могут встретиться только в тестовой выборке.

In [6]:
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

pipeline = Pipeline([
    ("preprocessor", preprocessor)
])

print("Pipeline создан успешно!")

Pipeline создан успешно!


### Проверка отсутствия утечки данных

Пайплайн обучается только на тренировочных данных с помощью fit_transform(). К тестовым данным применяется только transform(), без повторного обучения. Благодаря этому статистики тестовой выборки не используются при подготовке обучающей.

In [7]:
X_train_processed = pipeline.fit_transform(X_train)

X_test_processed = pipeline.transform(X_test)

print("Train после обработки:", X_train_processed.shape)
print("Test после обработки:", X_test_processed.shape)

Train после обработки: (712, 13)
Test после обработки: (179, 13)


In [8]:
print("Размер Train:", X_train_processed.shape)
print("Размер Test:", X_test_processed.shape)

assert X_train_processed.shape[1] == X_test_processed.shape[1]

print("Пропусков в Train:", np.isnan(X_train_processed).sum())
print("Пропусков в Test:", np.isnan(X_test_processed).sum())

assert not np.isnan(X_train_processed).any()
assert not np.isnan(X_test_processed).any()

print("Проверка пройдена!")

Размер Train: (712, 13)
Размер Test: (179, 13)
Пропусков в Train: 0
Пропусков в Test: 0
Проверка пройдена!


## Обоснование выбора стратегий

1. **Age:** пропущенные значения заполняются медианой, поскольку она устойчива к крайним значениям возраста.

2. **Fare:** используется медиана из-за наличия дорогих билетов, которые могут сильно влиять на среднее.

3. **SibSp и Parch:** применяется медиана как единая стратегия для числовых признаков. В данном датасете пропусков в них нет.

4. **Sex:** категории преобразуются в числовые столбцы с помощью OneHotEncoder.

5. **Pclass:** используется OneHotEncoder, чтобы модель могла учитывать каждый класс пассажиров отдельно.

6. **Embarked:** пропуски заполняются отдельной категорией Unknown, затем применяется OneHotEncoder.

7. **Cabin:** исключён из текущего пайплайна из-за большого количества пропусков и уникальных значений. В дальнейшем можно создать признак наличия информации о каюте.

### Защита от утечки данных

Сначала данные разделяются на обучающую и тестовую выборки. Затем пайплайн обучается только на X_train и применяется к X_test без повторного обучения. Это предотвращает использование статистик тестовой выборки при подготовке обучающих данных.